# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset on mass spectrometry analysis of extracellular vesicles isolated from human mast cells, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (Croissant schema)
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here we print all available record sets, the fields and columns they contain, referencing every element by its unique `@id`.

In [ ]:
# List all record sets, their fields and columns
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field @id: {field.id} (name: {getattr(field, 'name', None)}, dtype: {getattr(field, 'data_type', None)})")
    print(f"  Columns:")
    for col in getattr(rs, 'columns', []):
        print(f"    - Column @id: {col.id} (name: {getattr(col, 'name', None)})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview above.

In [ ]:
# We will extract all tabular record sets into DataFrames, using their @id as keys.

dataframes = {}

# List the record_set @id fields
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records in DataFrame for record set {record_set_id}")

# Print available columns for first record set if any
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"DataFrame columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
We'll perform common data processing steps: filtering, normalization, grouping, etc. In all cases, we reference fields by their `@id`.

In [ ]:
# For demonstration, select the first available DataFrame and numeric field

if dataframes:
    import numpy as np
    rs_id = list(dataframes.keys())[0]  # pick the first record set
    df = dataframes[rs_id]
    # Try to infer a numeric field (float or int) by column type or name hints
    numeric_field_id = None
    for col in df.columns:
        if (df[col].dtype in [np.float64, np.int64]) or 'abundance' in col.lower() or 'count' in col.lower() or 'value' in col.lower() or 'mw' in col.lower():
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        print(f"Using numeric field for EDA: {numeric_field_id}")
        # Drop rows with missing values in this field
        filtered_df = df[df[numeric_field_id].apply(lambda x: isinstance(x, (int, float, np.number)) or (isinstance(x, str) and x.replace('.', '', 1).isdigit()))]
        filtered_df[numeric_field_id] = filtered_df[numeric_field_id].astype(float)
        threshold = filtered_df[numeric_field_id].mean()
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_field]].head())

        # Try to group by a likely group field ('group', 'type', 'category', 'sample', etc)
        group_field = None
        for col in df.columns:
            if any(x in col.lower() for x in ['sample', 'group', 'category', 'class', 'type']):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df)
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between numeric fields in the record set. Adjust the plots below to match your chosen columns/fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    # Histogram of numeric values
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].astype(float), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group_field, plot the group means
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        group_means = df.groupby(group_field)[numeric_field_id].mean().reset_index()
        sns.barplot(x=group_field, y=numeric_field_id, data=group_means)
        plt.xticks(rotation=30)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load the FAIR² mass spectrometry dataset as defined by its Croissant schema, identified available record sets and their fields, loaded tabular data for analysis, and performed exploratory data analysis including filtering, normalization, grouping, and basic visualizations.

- All entities (record sets, fields, columns) were referenced by their `@id`s as required by the Croissant standard.
- This dataset enables secondary analyses, such as quantifying protein abundance and exploring variances across experimental groups.
- For deeper analysis, customize the EDA and visualization sections to specific biological questions or hypotheses.

For more, check the [mlcroissant documentation](https://mlcommons.github.io/croissant/).